# Input gender-recoverability probe

Evaluate original and blinded input representations with a train-fitted MLP classifier. This is information recoverability, not proof of its use by a recruitment model.

In [ ]:
import numpy as np
import os


from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

DATA_PATH = '../../data'
DB_FILE   = 'FairCVdb.npy'

# Configuration
HIDDEN_LAYERS = (40, 20)
MAX_ITER      = 500
SEED          = 32

In [ ]:
fairCV = np.load(os.path.join(DATA_PATH, DB_FILE), allow_pickle=True).item()

profiles_train = fairCV['Profiles Train']
profiles_test  = fairCV['Profiles Test']

# BERT embeddings
text_train_original = np.load(os.path.join(DATA_PATH, 'original_bios_train_bert.npy'))
text_test_original  = np.load(os.path.join(DATA_PATH, 'original_bios_test_bert.npy'))
text_train_blinded  = np.load(os.path.join(DATA_PATH, 'blinded_bios_train_bert.npy'))
text_test_blinded   = np.load(os.path.join(DATA_PATH, 'blinded_bios_test_bert.npy'))

print('profiles :', profiles_train.shape, profiles_test.shape)
print('original :', text_train_original.shape, text_test_original.shape)
print('blinded  :', text_train_blinded.shape, text_test_blinded.shape)

In [ ]:
def probe(X_train, X_test, y_train, y_test):
    """
    Predict gender from one input representation usign a classifier.

    Returns test-set accuracy. Close to 0.5 means the representation carries little
    recoverable gender information; close to 1.0 means gender is fully recoverable.

    We're only measuring recoverability, a number close to 1.0 does not mean the model will definitely use the gender information. It just means the information is available to be potentially used.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    clf = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        max_iter=MAX_ITER,
        random_state=SEED,
    )
    clf.fit(X_train_scaled, y_train)

    return accuracy_score(y_test, clf.predict(X_test_scaled))

In [ ]:
blocks = {
    'tabular original': (profiles_train[:, [1, 4, 5, 6, 7, 8, 9, 10]],
                         profiles_test[:, [1, 4, 5, 6, 7, 8, 9, 10]]),
    'tabular blinded':  (profiles_train[:, 4:11], profiles_test[:, 4:11]),
    'face biased': (profiles_train[:, 11:31], profiles_test[:, 11:31]),
    'face agnostic': (profiles_train[:, 31:51], profiles_test[:, 31:51]),
    'BERT original': (text_train_original, text_test_original),
    'BERT blinded': (text_train_blinded, text_test_blinded),
}

y_train = profiles_train[:, 1]
y_test  = profiles_test[:, 1]

male_count = np.count_nonzero(y_test == 0)
female_count = np.count_nonzero(y_test == 1)
total = y_test.size

print("male count:", male_count)
print("female count:", female_count)
print("total count:", total)

In [ ]:
chance = max(male_count/total, female_count/total)
accuracies = []

for block, (X_train, X_test) in blocks.items():
    accuracy = probe(X_train, X_test, y_train, y_test)
    accuracies.append(accuracy)
    print(f"{block}: accuracy = {accuracy}")

print(chance)

Interpret the resulting accuracy relative to the majority-class baseline. Original notebook commentary was removed from this export; consult the thesis for the reported analysis.